# LangChain ↔ Databricks — `ChatDatabricks` & `DatabricksVectorSearch`
**Roadmap:** Module 05 · Topic 05.2 · `[Hands-on]`

Build a minimal **RAG chain** for the *Unity Airways* assistant using the two first-party
LangChain–Databricks classes. Mirrors *Practical MLflow for GenAI on Databricks*, Ch4 (pp. 140–147).

### Prerequisites
- **Compute:** Serverless notebook compute (or an ML runtime, e.g. 15.4 LTS ML+).
- **Model:** a pay-per-token **Foundation Model API** endpoint (default: `databricks-claude-3-7-sonnet`).
- **Vector Search:** an **endpoint** + a **Delta-sync index** with Databricks-managed embeddings (see Module 04). You need the index's `catalog.schema.index` name.
- **Unity Catalog:** read access to the index and source docs.
- **Libraries:** `databricks-langchain`, `mlflow` (installed below).

## 0 · Install dependencies

In [ ]:
%pip install -U databricks-langchain mlflow langchain-core

In [ ]:
dbutils.library.restartPython()

## 1 · Configuration
Set these via the widgets at the top of the notebook (or edit the defaults). Fill in your Vector
Search endpoint + index before running the retrieval cells.

In [ ]:
dbutils.widgets.text("llm_endpoint", "databricks-claude-3-7-sonnet", "1 · LLM serving endpoint")
dbutils.widgets.text("vs_endpoint", "", "2 · Vector Search endpoint")
dbutils.widgets.text("vs_index", "", "3 · Vector Search index (catalog.schema.index)")

LLM_ENDPOINT = dbutils.widgets.get("llm_endpoint")
VS_ENDPOINT  = dbutils.widgets.get("vs_endpoint")
VS_INDEX     = dbutils.widgets.get("vs_index")

QUESTION = "How do I book flights with Unity Airways?"  # the book's running example query

print(f"LLM endpoint : {LLM_ENDPOINT}")
print(f"VS endpoint  : {VS_ENDPOINT or '(TODO: set the vs_endpoint widget)'}")
print(f"VS index     : {VS_INDEX or '(TODO: set the vs_index widget)'}")

## 2 · Enable MLflow tracing
One call and **every** LangChain invocation below is captured as an MLflow Trace
(view them in the experiment's **Traces** tab). Tracing is covered in Module 07.

In [ ]:
import mlflow

mlflow.langchain.autolog()  # auto-trace all LangChain calls in this notebook

## 3 · `ChatDatabricks` — call a served model (LLM-only baseline)
`ChatDatabricks` wraps a LangChain `ChatModel` pointing at a Model Serving / Foundation Model endpoint.
First, see what the LLM answers **without** any retrieved context.

In [ ]:
from databricks_langchain import ChatDatabricks

chat_model = ChatDatabricks(
    endpoint=LLM_ENDPOINT,
    temperature=0,
    max_tokens=256,
)

# LLM-only answer (not grounded in Unity Airways docs yet)
print(chat_model.invoke(QUESTION).content)

## 4 · `DatabricksVectorSearch` — query the index
`DatabricksVectorSearch` wraps a LangChain `VectorStore` over a Vector Search index.
Use `.similarity_search(...)` to inspect what gets retrieved.

In [ ]:
from databricks_langchain import DatabricksVectorSearch

assert VS_ENDPOINT and VS_INDEX, "Set the 'vs_endpoint' and 'vs_index' widgets first (Module 04 creates the index)."

vector_store = DatabricksVectorSearch(
    endpoint=VS_ENDPOINT,
    index_name=VS_INDEX,
)

# direct similarity search — peek at the top match
for doc in vector_store.similarity_search(query=QUESTION, k=1):
    print(f"* {doc.page_content}\n  metadata={doc.metadata}")

## 5 · Turn the index into a retriever
`.as_retriever(...)` returns the standard LangChain retriever interface, so it composes with any chain.

In [ ]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3, "query_type": "ANN"},
)

# the retriever returns LangChain Documents
print(retriever.invoke(QUESTION))

## 6 · Assemble the RAG chain (LCEL)
Compose `retriever → prompt → model → parser`. The prompt merges the retrieved **context** with the
user **question**, so the answer is grounded in Unity Airways' own docs.

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

prompt = PromptTemplate(
    template=(
        "You are a customer support assistant for Unity Airways.\n"
        "Answer the question using ONLY the context. If unsure, say you don't know.\n\n"
        "Context:\n{context}\n\n"
        "Question: {question}"
    ),
    input_variables=["context", "question"],
)


def format_docs(docs):
    """Flatten retrieved Documents into a single context string."""
    return "\n\n".join(d.page_content for d in docs)


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | chat_model
    | StrOutputParser()
)

## 7 · Invoke — grounded answer

In [ ]:
print(rag_chain.invoke(QUESTION))

## ✅ Recap & next steps
- **`ChatDatabricks(endpoint=...)`** → call any served / Foundation Model endpoint from LangChain.
- **`DatabricksVectorSearch(index_name=...)`** → query a Vector Search index; `.as_retriever()` makes it chain-ready.
- **LCEL** wires `retriever → prompt → model → parser`; `mlflow.langchain.autolog()` traces every run.

**Good practice (Module 05.5–05.7):** move `LLM_ENDPOINT`, `VS_INDEX`, and the prompt into a
`rag_chain_config.yml` loaded via `mlflow.models.ModelConfig`, then **log the chain as code**.

**Next roadmap topic:** `05.3 — LLM-only app → full RAG chain`.